In [1]:
# Purpose: Load order dates for holiday feature engineering.

from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

if not (ROOT / "data" / "processed").is_dir():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

orders = pd.read_csv(
    DATA_DIR / "research_data.csv",
    usecols=[
        "order_id",
        "order_purchase_timestamp",
        "order_estimated_delivery_date"
    ],
    dtype={"order_id": "string"}
)

date_columns = [
    "order_purchase_timestamp",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="raise"
    )

assert orders["order_id"].notna().all()
assert orders["order_id"].is_unique

display(orders.head())

print("Total orders:", len(orders))
print("Missing dates:")
print(orders[date_columns].isna().sum())

print(
    "Earliest purchase:",
    orders["order_purchase_timestamp"].min()
)

print(
    "Latest purchase:",
    orders["order_purchase_timestamp"].max()
)

print(
    "Latest estimated delivery:",
    orders["order_estimated_delivery_date"].max()
)

,order_id,order_purchase_timestamp,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,2017-10-02 10:56:33,2017-10-18
1,53cdb2fc8bc7dce0b6741e2150273451,2018-07-24 20:41:37,2018-08-13
2,47770eb9100c2d0c44946d9cf07ec65d,2018-08-08 08:38:49,2018-09-04
3,949d5b44dbf5de918fe9c16f97b45f8a,2017-11-18 19:28:06,2017-12-15
4,ad21c59c0840e6cb83a9ceb5573f8159,2018-02-13 21:18:39,2018-02-26


Total orders: 99441
Missing dates:
order_purchase_timestamp         0
order_estimated_delivery_date    0
dtype: int64
Earliest purchase: 2016-09-04 21:15:19
Latest purchase: 2018-10-17 17:30:18
Latest estimated delivery: 2018-11-12 00:00:00


In [ ]:
# Purpose: Load and validate the holiday calendar.

holidays = pd.read_csv(
    DATA_DIR.parent / "external" / "holidays" / "brazil_national_holidays.csv",
    dtype="string"
)

holidays["holiday_date"] = pd.to_datetime(
    holidays["holiday_date"],
    format="%Y-%m-%d",
    errors="raise"
)

holidays = (
    holidays.sort_values("holiday_date")
    .reset_index(drop=True)
)

assert holidays["holiday_date"].notna().all()
assert holidays["holiday_date"].is_unique

assert holidays["holiday_name_en"].notna().all()
assert holidays["source_url"].notna().all()

assert holidays["holiday_type"].eq(
    "national_as_listed_in_federal_calendar"
).all()

assert (
    holidays["holiday_date"].dt.year
    == pd.to_numeric(holidays["source_year"], errors="raise")
).all()

holiday_counts = (
    holidays.groupby(holidays["holiday_date"].dt.year)
    .size()
    .rename("holiday_count")
)

assert holiday_counts.to_dict() == {
    2016: 9,
    2017: 9,
    2018: 9
}

display(holidays.head())
display(holiday_counts.to_frame())

print("Holiday calendar checks passed.")

,holiday_date,holiday_name,holiday_name_en,holiday_type,source_year,source_url
0,2016-01-01,Confraternização Universal,New Year's Day,national_as_listed_in_federal_calendar,2016,https://progep.ufc.br/pt/mpog-divulga-calendar...
1,2016-03-25,Paixão de Cristo,Good Friday,national_as_listed_in_federal_calendar,2016,https://progep.ufc.br/pt/mpog-divulga-calendar...
2,2016-04-21,Tiradentes,Tiradentes Day,national_as_listed_in_federal_calendar,2016,https://progep.ufc.br/pt/mpog-divulga-calendar...
3,2016-05-01,Dia Mundial do Trabalho,Labour Day,national_as_listed_in_federal_calendar,2016,https://progep.ufc.br/pt/mpog-divulga-calendar...
4,2016-09-07,Independência do Brasil,Brazilian Independence Day,national_as_listed_in_federal_calendar,2016,https://progep.ufc.br/pt/mpog-divulga-calendar...


,holiday_count
holiday_date,
2016,9
2017,9
2018,9


Holiday calendar checks passed.


In [3]:
# Purpose: Create holiday features for each order.

import numpy as np

purchase_date = orders["order_purchase_timestamp"].dt.normalize()
estimated_date = orders["order_estimated_delivery_date"].dt.normalize()

# This calendar covers full calendar years 2016-2018.
calendar_start = pd.Timestamp("2016-01-01")
calendar_end = pd.Timestamp("2018-12-31")

assert purchase_date.notna().all()
assert purchase_date.between(calendar_start, calendar_end).all()

holiday_dates = holidays["holiday_date"].to_numpy(
    dtype="datetime64[ns]"
)

purchase_values = purchase_date.to_numpy(dtype="datetime64[ns]")
estimated_values = estimated_date.to_numpy(dtype="datetime64[ns]")

order_holidays = orders[["order_id"]].copy()

# Feature 1: Is the purchase date a selected holiday?
order_holidays["purchase_is_national_holiday"] = (
    purchase_date.isin(holidays["holiday_date"]).astype("Int64")
)

# Feature 2: Days until the next holiday, including the purchase date.
next_position = np.searchsorted(
    holiday_dates, purchase_values, side="left"
)

has_next_holiday = next_position < len(holiday_dates)

days_to_next = pd.Series(
    pd.NA, index=orders.index, dtype="Int64"
)

days_to_next.loc[has_next_holiday] = (
    (
        holiday_dates[next_position[has_next_holiday]]
        - purchase_values[has_next_holiday]
    ) / np.timedelta64(1, "D")
).astype(int)

order_holidays["days_to_next_national_holiday"] = days_to_next

# Feature 3: Holiday count in (purchase date, estimated date].
valid_window = (
    estimated_date.notna()
    & estimated_date.ge(purchase_date)
    & estimated_date.le(calendar_end)
)

holiday_count = pd.Series(
    pd.NA, index=orders.index, dtype="Int64"
)

start_position = np.searchsorted(
    holiday_dates, purchase_values[valid_window], side="right"
)

end_position = np.searchsorted(
    holiday_dates, estimated_values[valid_window], side="right"
)

holiday_count.loc[valid_window] = end_position - start_position

order_holidays["national_holiday_count_in_estimated_window"] = (
    holiday_count
)

display(order_holidays.head(10))

,order_id,purchase_is_national_holiday,days_to_next_national_holiday,national_holiday_count_in_estimated_window
0,e481f51cbdc54678b7cc49136f2d6af7,0,10,1
1,53cdb2fc8bc7dce0b6741e2150273451,0,45,0
2,47770eb9100c2d0c44946d9cf07ec65d,0,30,0
3,949d5b44dbf5de918fe9c16f97b45f8a,0,37,0
4,ad21c59c0840e6cb83a9ceb5573f8159,0,45,0
5,a4591c265e18cb1dcee52889e2d8acc3,0,60,0
6,136cce7faa42fdb2cefd53fdc79a6098,0,3,3
7,6514b8ad8028c9f2cc2374ded245783f,0,114,0
8,76c6e866289321a7c93b82b54852dc33,0,81,0
9,e69bfb5eb88e0ed6a785585b27e16dbf,0,40,0


In [4]:
# Purpose: Validate and export order-level holiday features.

assert len(order_holidays) == len(orders)
assert order_holidays["order_id"].is_unique
assert order_holidays["order_id"].equals(orders["order_id"])

holiday_features = [
    "purchase_is_national_holiday",
    "days_to_next_national_holiday",
    "national_holiday_count_in_estimated_window"
]

assert order_holidays[
    "purchase_is_national_holiday"
].isin([0, 1]).all()

for column in holiday_features[1:]:
    assert order_holidays[column].dropna().ge(0).all()

on_holiday = order_holidays["purchase_is_national_holiday"].eq(1)

assert order_holidays.loc[
    on_holiday, "days_to_next_national_holiday"
].eq(0).all()

display(pd.DataFrame({
    "missing_rows": order_holidays[holiday_features].isna().sum(),
    "missing_percent": (
        order_holidays[holiday_features].isna().mean() * 100
    ).round(3)
}))

output_path = DATA_DIR / "order_holiday_features.csv"
order_holidays.to_csv(output_path, index=False)

print("All holiday checks passed.")
print("Saved:", output_path)
print("Dataset shape:", order_holidays.shape)

,missing_rows,missing_percent
purchase_is_national_holiday,0,0.0
days_to_next_national_holiday,0,0.0
national_holiday_count_in_estimated_window,0,0.0


All holiday checks passed.
Saved: c:\Users\LENOVO\OneDrive\Desktop\PTTK\Group5_STAT3013.R11\data\processed\order_holiday_features.csv
Dataset shape: (99441, 4)
